In [2]:
%pip install numpy trimesh
import numpy as np
import trimesh
import matplotlib.pyplot as plt
from scipy.spatial import Delaunay
import time


Note: you may need to restart the kernel to use updated packages.


# Poisson Equation in 2D
$$-\Delta_\Gamma u=f \quad \text{(Laplace Beltrami)}$$


## Weak formulation
$u\in C^{2}(\Omega) \rightarrow u \in C^1(\Omega)$

$$$$

$$-\int_\Gamma v \Delta_\Gamma u \hspace{1mm} dS=\int_\Gamma fv \hspace{1mm} dS$$

$$$$

$$\int_\Gamma \nabla_\Gamma u \cdot \nabla_\Gamma v \hspace{1mm} dS - \int_{\partial \Gamma} \frac{\partial u}{\partial \nu} v \hspace{1mm} ds =\int_\Omega fv \hspace{1mm} dS \quad \text{(Divergence Theorem)}$$


# Mesh Solution

In a mesh our domain is partitioned into triangular meshes, where each vertex has its own $u$ value. Each point on the surface can then be approximated by an interpolation within the triangular face (element)

## Basis Function

The basis function for a vertex is a pyramid structure where the tip is the vertex itself and the base is the adjacent vertices (linear lagrange)

$$\Psi_i = \bigg\{ \begin{matrix} 1 & \text{vertex i} \\ 0 & \text{not vertex i}\end{matrix}$$

## Discretization

$$u_h(x,y)=\sum_{i}^NU_i\Psi_i(x,y)$$
$$$$
$$N = \text{total number of vertices}$$
$$U_i = \text{value of u at vertex i}$$
$$\Psi_i = \text{basis function of vertex i}$$

$$$$

 $u=u_h$ and $v=\sum_i \Psi_i \quad \text{(Galerkin Method)} $

$$$$
Ignoring boundary conditions, we get a system of equations involving a stiffness matrix (A) and a load vector (b)

$$$$

$$\int_\Gamma \left(\sum_i^N u_i\nabla_\Gamma \Psi_i\right) \left(\sum_j^N \nabla_\Gamma \Psi_j\right) d S =\int_\Gamma \sum_i^N \Psi_i f \hspace{1mm}
dS $$

$$$$

$$A_{i,j} = \int_{\Gamma}\nabla_\Gamma \Psi_i \cdot \nabla_\Gamma \Psi_j \hspace{1mm} dS$$
$$b_{i} = \int_{\Gamma} \Psi_i f \hspace{1mm} dS$$

$$$$

$$AU=b, \quad U_i=u_i$$
$$$$
$$A\in \mathbb{R}^{N \times N},\quad b\in \mathbb{R}^{N},\quad U\in \mathbb{R}^{N}$$

# Elements

Given a triangle with vertices $p_0,p_1,p_2$, we first compute the edges $e_1=p_1-p_0, e_2=p_2-p_0$

The normal vector is $N =e_1 \times e_2, \quad n = \frac{N}{|N|}$

The triangle area is $E_A =\frac{1}{2} |N|$

## Gradient

The gradient of $\Psi_i$ is a constant vector that must lie in the triangle plane and be perpendicular to the opposing edge (0 gradient across the opposing edge). Since $\Psi_i$ changes from 0 to 1 across the height of the triangle $h_0$ (with respect to vertex and the opposite edge), we expect the gradient to have a magnitude of $\frac{1}{h_0}$. Combining these geometric criterion gets us the following gradients:

$\nabla_\Gamma \Psi_0 = \frac{n \times (p_2-p_1)}{2A}$
$$$$
$\nabla_\Gamma \Psi_1 = \frac{n \times (p_0-p_2)}{2A}$
$$$$
$\nabla_\Gamma \Psi_2 = \frac{n \times (p_1-p_0)}{2A}$




In [22]:
class Element():
    def __init__(self, Vertices, face):
        self.face = face
        self.p = Vertices[face] # 3x3 matrix of vertices
        
        # Compute edges and face normal
        self.e = [self.p[1] - self.p[0], self.p[2] - self.p[0]]
        self.N = np.cross(self.e[0], self.e[1])
        self.n = self.N / np.linalg.norm(self.N)
        self.A = np.linalg.norm(self.N) / 2.0
        
        M = np.vstack([self.e[0], self.e[1], self.n])
        
        M_inv = np.linalg.inv(M)
        
        inv_e1 = M_inv[:, 0]
        inv_e2 = M_inv[:, 1]
        
        grad_0 = - inv_e1 - inv_e2
        grad_1 = inv_e1
        grad_2 = inv_e2
        
        self.grad = [grad_0, grad_1, grad_2]

#  Local Stiffness Matrix

$$A_{i,j}^e=\int_{\Gamma_e}\nabla_\Gamma \Psi_i \cdot \nabla_\Gamma \Psi_j dS = \left(\nabla_\Gamma \Psi_i \cdot \nabla_\Gamma \Psi_j\right) E_a \quad \text{Surface is Constant Triangle} $$

$$$$

$$A^e=\begin{bmatrix}
A_{0,0} & A_{0,1} & A_{0,2} \\
A_{1,0} & A_{1,1} & A_{1,2} \\
A_{2,0} & A_{2,1} & A_{2,2} \\
\end{bmatrix}$$

$$$$

Note that these indices are local, they each represent a global index from $0...N$

# Global Stiffness Matrix
for each vertex add all the local entries to their corresponding global entry


In [4]:
def update_global_stiffness(A,Vertices,face):
  element = Element(Vertices,face)
  for i in range(3):
    for j in range(3):
      A[face[i],face[j]] += element.grad[i].dot(element.grad[j])*element.A


def create_global_stiffness(A,Vertices,Faces):
  for face in Faces:
    update_global_stiffness(A,Vertices,face)



# Local Load Vector

$$b_i^e = \int_{\Gamma_e} f \Psi_i \hspace{1mm} dS$$

## Centroid Quadrature

The easiest way to approximate is to consider $f$ at the centroid
$$$$
$$p_c=\frac{p_0+p_1+p_2}{3} \rightarrow f_c = f(p_c)$$
$$$$
note that the area across one basis function is $\frac{1}{3}$ the area of total triangle
$$$$
$$b_i^e = \frac{Af_c}{3}\begin{bmatrix}1 \\ 1 \\1 \end{bmatrix}$$

## Face Values

if $f$ is known per face ($f_e$) then you can use the face value instead of the centroid value
$$$$
$$b_i^e = \frac{Af_e}{3}\begin{bmatrix}1 \\ 1 \\1 \end{bmatrix}$$


## Vertex Values

A more accurate approximation would be to utilize a linear combination of the basis functions for f.
$$$$
$$f_h = f_0\Psi_0 + f_1\Psi_1 + f_2 \Psi_2 \quad f_i = f(p_i)$$
$$$$

note that the area of across the product of basis functions can is an even distribution of the area

$$$$

$$\int_{\Gamma_e}\Psi_i \Psi_j \hspace{1mm} dS = \bigg \{ \begin{matrix} \frac{A}{6}& i = j \\ \frac{A}{12} & i \neq j\end{matrix}$$

$$$$

$$b_i^e = \frac{A}{12}\begin{bmatrix}2f_0 + f_1 + f_2 \\ f_0 + 2f_1 + f_2 \\f_0 + f_1 + 2f_2 \end{bmatrix}$$

# Global Load Vector
for each vertex add all the local entries to their corresponding global entry

In [18]:
def LV_Centroid_Quadrature(b,Vertices,face):
  element = Element(Vertices,face)
  p_c = (element.p[0]+element.p[1]+element.p[2])/3
  f_c = f(p_c[0],p_c[1],p_c[2])
  A = element.A
  for i in range(3):
    b[face[i]] += A/3*f_c

def LV_Face_Values(b,face):
  element = Element(face)
  f_e = f(face)
  A = element.A
  for i in range(3):
    b[face[i]] += A/3 * f_e


def LV_Vertex_Values(b, f, Vertices, face):
    element = Element(Vertices, face)
    A = element.A
    
    # Evaluate the source term f at each of the 3 vertices
    f0 = f(element.p[0][0], element.p[0][1], element.p[0][2])
    f1 = f(element.p[1][0], element.p[1][1], element.p[1][2])
    f2 = f(element.p[2][0], element.p[2][1], element.p[2][2])
    
    # Correct linear finite element load vector integration scaling
    b_local = (A / 12.0) * np.array([
        2.0 * f0 + f1 + f2,
        f0 + 2.0 * f1 + f2,
        f0 + f1 + 2.0 * f2
    ])
    
    for i in range(3):
        b[face[i]] += b_local[i]

def create_global_load(b,f,Vertices,Faces):
  for face in Faces:
    LV_Vertex_Values(b,f,Vertices,face)



# Determining Boundary

If an edge is a boundary, then that edge should only appear in one element. To determine the edges simple find each unique edge. You can then determine the boudary vertices directly from the boundary edges.

$$$$

We need the boundary to apply the following boundary condition term:
$$$$
$$\int_{\partial \Gamma} \frac{\partial u}{\partial \nu} v \hspace{1mm} ds$$

In [6]:
def get_Boundary(Faces):
  boundary_edges = []
  edge_count = {}
  boundary_vertices = []

  #get edge counts
  for face in Faces:
    e_1 = tuple(sorted([face[0],face[1]]))
    e_2 = tuple(sorted([face[1],face[2]]))
    e_3 = tuple(sorted([face[2],face[0]]))
    e = [e_1,e_2,e_3]
    for edge in e:
      if edge not in edge_count:
        edge_count[edge] = 1
      else:
        edge_count[edge] += 1

  #get bounded edges:
  for edge in edge_count:
    if edge_count[edge] == 1:
      boundary_edges.append(edge)

  #get bound vertices
  for edge in boundary_edges:
    for vertex in edge:
      if vertex not in boundary_vertices:
        boundary_vertices.append(vertex)


  return boundary_vertices,boundary_edges


# Dirichlet Boundary Condition

$$u=g \quad \text{on} \hspace{3mm}\Gamma_d$$

This means for every boundary ($d$), $U_d=g(p_d)=g_d$

We need to reset all the boundary values and apply the boundary value term in the weak formulation
$$$$

1. Subtract known contributions for all rows ($i$) $F_i \leftarrow F_i - A_{i,d}\hspace{1mm} g_d$

2. Zero out the row and column of boundary $A_{i,d} = A_{d,i}= 0$

3. Set diagonal entry to one $A_{d,d}=1$

4. set load vector entry to g $F_d=g_d$

$$$$

These steps ensure that $U_d = g_d$ and the boundary value term is added

In [7]:
def apply_dirichlet(A,b,Vertices,boundary_vertices,g):
  for d in boundary_vertices:

    #apply boundary term
    for i in range(len(Vertices)):
      A_temp = A[i][d]
      if A_temp == 0 :
        continue
      b[i] -= A_temp*g(Vertices[d][0], Vertices[d][1], Vertices[d][2])

    #apply boundary condition
    for i in range(len(Vertices)):
      A[d,i] = 0
      A[i,d] = 0

    A[d,d] = 1
    b[d] = g(Vertices[d][0], Vertices[d][1], Vertices[d][2])


# Neumann Boundary Conditions

$$\frac{\partial u}{\partial \nu}=h \quad \text{on} \hspace{3mm}\Gamma_N$$

$$$$

We can directly evaluate the boundary term integral, for an edge element this line integral is simply $hL$ where $L$ is the length of the edge. We just need to split this and add it to the load vector corresponding to the vertices of the edge

$$$$

$$b_i \leftarrow b_i + \frac{hL}{2} \quad b_j \leftarrow b_j + \frac{hL}{2}$$

In [8]:
def apply_neumann(A,b,Vertices,boundary_edges,h):
  for edge in boundary_edges:
    i = edge[0]
    j = edge[1]
    L = np.linalg.norm(Vertices[i]-Vertices[j])
    X,Y,Z = 0.5*(Vertices[i]+Vertices[j])
    H = h(X,Y,Z)
    b[i] += H*L/2
    b[j] += H*L/2

# Robin Boundary Conditions

$$\frac{\partial u}{\partial \nu} + \alpha u = r \quad \text{on} \hspace{3mm} \Gamma_R$$

$$$$

Considering the edge we can construct a local stiffness matrix and load vector

$$$$

$$A^e=\frac{\alpha L}{6} \begin{bmatrix}  2 & 1 \\ 1 & 2\end{bmatrix}$$

$$$$

$$b^e=\frac{rL}{2} \begin{bmatrix} 1 \\ 1 \end{bmatrix}$$

$$$$

To apply the boundary condition we just need to update the global stiffness matrix and load vector with the corresponding local one for each boundary edge

In [9]:
def apply_robin(A,b,Vertices,boundary_edges,alpha,r):
  for edge in boundary_edges:
    i = edge[0]
    j = edge[1]
    X,Y,Z = 0.5*(Vertices[i]+Vertices[j])
    R = r(X,Y,Z)
    Alpha = alpha(X,Y,Z)
    L = np.linalg.norm(Vertices[i]-Vertices[j])
    A_local = Alpha*L/6 * np.array([[2,1],[1,2]])
    b_local = R*L/2 * np.array([1,1])

    for k in range(2):
      for l in range(2):
        A[i,j] += A_local[k,l]
        A[j,i] += A_local[k,l]

    b[i] += b_local[k]
    b[j] += b_local[k]

# Solving U

$$U = A^{-1} b$$

The final result will be a vector holding all the u values at each vertex

$$U=\begin{bmatrix}u_0 \\u_1 \\ \vdots \\ u_N\end{bmatrix}$$

In [10]:
%%script false --no-raise-error

#Load mesh
Mesh = trimesh.load('./meshes/ripple.obj')
Vertices,Faces = Mesh.vertices, Mesh.faces
N = len(Vertices)

#Initialize System
A = np.zeros((N,N))
b = np.zeros(N)

#Load System
create_global_stiffness(A,Vertices,Faces)
create_global_load(b,Vertices,Faces)

#Get Bounds
boundary_vertices,boundary_edges = get_Boundary(Faces)

#Define Boundary Conditions
def g(x,y,z):
  return x**2

def h(x,y,z):
  return 0

def alpha(x,y,z):
  return 1

def r(x,y,z):
  return 0

#Apply Boundary Conditions

#apply_dirichlet(A,b,Vertices,boundary_vertices,g)
#apply_neumann(A,b,Vertices,boundary_edges,h)
apply_robin(A,b,Vertices,boundary_edges,alpha,r)

#Solve
U = np.linalg.solve(A,b)

#Save Results
File = open('./solutions/ripple.csv','w')
File.write('x,y,z,u\n')

for i in range(N):
  (x,y,z) = Vertices[i]
  File.write(str(x)+', '+str(y)+', '+str(z)+', '+str(U[i])+'\n')

File.close()

Couldn't find program: 'false'


# Testing

In order to test the validity of our FEM we will consider comparing it to known solutions on the plane.

$$u(x,y)=(x^2-1)(y^2-1) \in \Gamma [-1,1] \times [-1,1]$$
$$-\Delta u = -2(x^2+y^2-2)$$
$$u(\Gamma_d) = 0 \Rightarrow g=0$$

In [ ]:
X = np.linspace(-1,1,3)
Y = np.linspace(-1,1,3)

domain = np.array([[x, y] for x in X for y in Y])
tri = Delaunay(domain)

faces = tri.simplices
vertices = np.array([[x,y,0] for x in X for y in Y])
mesh  = trimesh.Trimesh(vertices,faces)

for i in range(1):
    mesh = mesh.subdivide()
    
    current_vertices = mesh.vertices
    updated_vertices = []
    for p in current_vertices:
        x, y = p[0], p[1]
        z = x**2 + y**2  # Force the vertex onto the paraboloid geometry
        updated_vertices.append([x, y, z])
    
    # Re-assign the snapped vertices back to the mesh object
    mesh.vertices = np.array(updated_vertices)

# Proceed to assign your final Vertices and Faces for the FEM loops
Vertices = mesh.vertices
Faces = mesh.faces

#source term
def f(x, y, z):
    W_sq = 1.0 + 4.0 * x**2 + 4.0 * y**2
    term1 = -2.0 * (y**2 - 1.0) - 2.0 * (x**2 - 1.0)
    term2 = (8.0 / W_sq) * (x**2 * (y**2 - 1.0) + y**2 * (x**2 - 1.0))
    return (term1 + term2) / W_sq

N = len(Vertices)

#Initialize System
A = np.zeros((N,N))
b = np.zeros(N)

start_time = time.perf_counter()

#Load System
create_global_stiffness(A,Vertices,Faces)
create_global_load(b,f,Vertices,Faces)

#Get Bounds
boundary_vertices,boundary_edges = get_Boundary(Faces)


#Define Boundary Conditions
def g(x,y,z):
  return 0

def h(x,y,z):
  return 0

def alpha(x,y,z):
  return 1

def r(x,y,z):
  return 0

#Apply Boundary Conditions

apply_dirichlet(A,b,Vertices,boundary_vertices,g)
#apply_neumann(A,b,Vertices,boundary_edges,h)
#apply_robin(A,b,Vertices,boundary_edges,alpha,r)

#Solve
U = np.linalg.solve(A,b)

end_time = time.perf_counter()
print(f"Time taken to solve the system: {end_time - start_time:.6f} seconds")

#Save Results
File = open('./solutions/parab-quad/.csv','w')
File.write('x,y,u\n')

print(len(U))

boundary_set = set(boundary_vertices)

for i in range(N):
    (x, y, z) = Vertices[i]
    if i in boundary_set:
        File.write(f"{x}, {y}, {z}, {U[i]}, boundary\n")
    else:
        File.write(f"{x}, {y}, {z}, {U[i]}\n")

File.close()


Time taken to solve the system: 0.002737 seconds
9
